No,１

In [ ]:
import os
import shutil
import csv
import re
from pathlib import Path
import json


# --- 共通の復元基盤 undo_utils.py を読み込む ---
# ※ このセルの import を並べ替えても壊れないように、undo_utils だけは
#    import 文ではなく importlib 経由で読み込んでいます。
import importlib
import sys
from pathlib import Path


def _locate_undo_utils():
    """undo_utils.py があるフォルダを探す（VS Code の作業ディレクトリ設定に依存しない）。"""
    candidates = []
    # 1) VS Code がノートブック自身のパスを教えてくれる場合
    nb_file = globals().get("__vsc_ipynb_file__")
    if nb_file:
        candidates.append(Path(nb_file).parent)
    # 2) 作業ディレクトリと、その中／親の「コードフォルダ」
    cwd = Path.cwd()
    candidates += [cwd, cwd / "コードフォルダ", cwd.parent, cwd.parent / "コードフォルダ"]

    for c in candidates:
        if (c / "undo_utils.py").is_file():
            return c.resolve()

    raise FileNotFoundError(
        "undo_utils.py が見つかりません。\n"
        "このノートブックと同じ「コードフォルダ」内に undo_utils.py があるか確認してください。\n"
        f"探した場所: {[str(c) for c in candidates]}"
    )


_uu_dir = str(_locate_undo_utils())
if _uu_dir not in sys.path:
    sys.path.insert(0, _uu_dir)

uu = importlib.import_module("undo_utils")
importlib.reload(uu)  # undo_utils.py を編集した場合も反映されるようにする

In [ ]:
# 選択したメインフォルダ内のサブフォルダ名を射出速度に変更
# （変更内容は _undo/undo_log.json に記録され、末尾の復元セルで元に戻せます）

# === 新しい名前のリスト ===
new_names = ["003", "005","010", "020", "040", "080", "120","160", "200", "240", "280", "320"]
# =========================

STEP_NAME = "01_サブフォルダ名の変更"


def rename_subfolders_with_log(names_list):
    target_dir = uu.select_folder("対象のメインフォルダを選択してください")
    if target_dir is None:
        return

    # _undo / _trash は名前変更の対象にしない
    subfolders = sorted(
        [f for f in target_dir.iterdir()
         if f.is_dir() and not uu.is_reserved(f, target_dir)],
        key=lambda x: x.name,
    )

    if len(subfolders) != len(names_list):
        print(f"エラー: サブフォルダの数（{len(subfolders)}個）と、"
              f"指定した新しい名前の数（{len(names_list)}個）が一致しません。")
        return

    print("-" * 40)

    with uu.UndoJournal(target_dir, STEP_NAME) as j:
        for folder, new_name in zip(subfolders, names_list):
            new_folder_path = folder.with_name(new_name)

            if new_folder_path.exists():
                print(f"スキップ: '{new_name}' は既に存在するため変更を見送りました。")
            else:
                original_name = folder.name
                j.move(folder, new_folder_path)
                print(f"変更完了: '{original_name}' -> '{new_name}'")

    print("-" * 40)


# 実行
rename_subfolders_with_log(new_names)

---
### ⏪ 復元（元に戻す）

このセルを実行すると、**このノートブックで行った直前の1工程**を巻き戻します。
（メインフォルダの `_undo/undo_log.json` に記録された履歴を使います）

繰り返し実行すれば、01〜06 のどの工程まででもさかのぼれます。
削除したファイルは `_trash` フォルダに退避されているので、これも一緒に元の場所へ戻ります。

In [ ]:
# ===== 共通の復元セル =====
# 直前に実行した1工程を巻き戻します。
# 続けて実行すれば、さらに1つ前の工程へとさかのぼれます。

uu.undo_interactive()